In [6]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
!nvidia-smi
!cp "/content/drive/MyDrive/taco_colab.zip" /content/
!unzip -q /content/taco_colab.zip -d /content/
!pip -q install ultralytics

Sat Sep 26 11:08:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P0             29W /   70W |    2795MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [10]:
from pathlib import Path
import yaml

root = Path("/content/taco_dataset")
p = root / "data.yaml"

data = yaml.safe_load(p.read_text())
data.update({
    "path": str(root),
    "train": "images/train",
    "val": "images/val",
})
p.write_text(yaml.safe_dump(data, allow_unicode=True))

for split in ("train", "val"):
    folder = root / data[split]
    images = [
        f for f in folder.glob("*")
        if f.suffix.lower() in (".jpg", ".jpeg", ".png")
    ]
    print(f"{split}: {len(images)}枚")

print(p.read_text())

train: 1200枚
val: 300枚
names:
  0: litter
path: /content/taco_dataset
train: images/train
val: images/val



In [12]:
from pathlib import Path
from shutil import copy2
from google.colab import drive
from ultralytics import YOLO

drive.mount("/content/drive")

# 以前のバックアップと分けて保存
backup = Path("/content/drive/MyDrive/taco_checkpoints_colab")
backup.mkdir(parents=True, exist_ok=True)

def save_to_drive(trainer):
    for name in ("last", "best"):
        src = Path(getattr(trainer, name))
        if src.is_file():
            copy2(src, backup / f"{name}.pt")

model = YOLO("/content/runs/detect/taco/weights/last.pt")
model.add_callback("on_model_save", save_to_drive)

model.train(
    resume=False,
    data="/content/taco_dataset/data.yaml",
    epochs=39,
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    project="/content/runs/detect",
    name="taco_colab",
    save=True,
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Ultralytics 8.4.163 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/taco_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=39, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7b9273988670>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 